# LAB 2: TUYỂN SINH ĐẠI HỌC - WEEK 4

## Mục lục
1. **Xử lý dữ liệu cơ bản**
2. **Feature Engineering**
3. **Trực quan hóa dữ liệu**
4. **Mô tả dữ liệu định lượng**
5. **Phân tích phân phối dữ liệu**
6. **Phân tích tương quan**

---


## 1. Xử lý dữ liệu cơ bản

### 1.1. Import các thư viện cần thiết


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Thiết lập style cho matplotlib
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Đã import thành công các thư viện cần thiết!")


### 1.2. Đọc dữ liệu


In [ ]:
# Đọc file dữ liệu tuyển sinh đại học
try:
    df = pd.read_csv('dulieuxettuyendaihoc.csv', header=0, delimiter=',', encoding='utf-8')
    print("Đã đọc thành công file 'dulieuxettuyendaihoc.csv'")
except FileNotFoundError:
    print("Không tìm thấy file 'dulieuxettuyendaihoc.csv'")
    print("Tạo dữ liệu mẫu để demo...")
    
    # Tạo dữ liệu mẫu
    np.random.seed(42)
    n_students = 500
    
    df = pd.DataFrame({
        'T5': np.random.uniform(3.0, 10.0, n_students),  # Điểm toán lớp 12
        'T6': np.random.uniform(3.0, 10.0, n_students),  # Điểm toán lớp 12
        'GT': np.random.choice(['Nam', 'Nữ'], n_students),  # Giới tính
        'DT': np.random.choice(['Kinh', 'Tày', 'Mường', 'Khmer'], n_students, p=[0.8, 0.1, 0.05, 0.05]),  # Dân tộc
        'KV': np.random.choice(['KV1', 'KV2', 'KV3'], n_students, p=[0.3, 0.4, 0.3]),  # Khu vực
        'KT': np.random.choice(['A', 'A1', 'B', 'C', 'D1'], n_students, p=[0.3, 0.2, 0.2, 0.15, 0.15]),  # Khối thi
        'NGONNGU': np.random.uniform(0, 10, n_students),  # Điểm ngôn ngữ
        'TOANLOGICPHANTICH': np.random.uniform(0, 10, n_students),  # Điểm logic
        'GIAIQUYETVANDE': np.random.uniform(0, 10, n_students),  # Điểm giải quyết vấn đề
        'NGAYTHI': np.random.choice([2020, 2021, 2022], n_students),  # Năm thi
        'DINHHUONGNGHENGHIEP': np.random.uniform(0, 10, n_students)  # Định hướng nghề nghiệp
    })

# Đọc 5 dòng dữ liệu đầu tiên
print("5 dòng dữ liệu đầu tiên:")
print(df.head())

# Xem thông tin tổng quan
print(f"\nThông tin tổng quan:")
print(f"Kích thước dữ liệu: {df.shape}")
print(f"Thông tin chi tiết:")
print(df.info())


### 1.3. Lấy thông tin các cột và đổi tên


In [ ]:
# Lấy thông tin các cột cần thiết
df = df[['T5', 'T6', 'GT', 'DT', 'KV', 'KT', 'NGONNGU', 'TOANLOGICPHANTICH', 
         'GIAIQUYETVANDE', 'NGAYTHI', 'DINHHUONGNGHENGHIEP']]

# Đổi tên cột
df.rename(columns={
    'TOANLOGICPHANTICH': 'LOGIC',
    'GIAIQUYETVANDE': 'UNGXU',
    'DINHHUONGNGHENGHIEP': 'HUONGNGHIEP'
}, inplace=True)

print("Dữ liệu sau khi đổi tên cột:")
print(df.head())

print(f"\nCác cột hiện tại: {list(df.columns)}")


### 1.4. Xử lý dữ liệu thiếu


In [ ]:
# Xóa bỏ các dòng dữ liệu rỗng
print("Dữ liệu trước khi xóa dòng rỗng:")
print(f"Số dòng: {len(df)}")

df.dropna(how='all', inplace=True)
print(f"\nDữ liệu sau khi xóa dòng rỗng:")
print(f"Số dòng: {len(df)}")

# Dùng heatmap để trực quan dữ liệu bị thiếu
plt.figure(figsize=(10, 6))
sns.heatmap(df.isna().transpose(), cmap='YlGnBu', cbar_kws={'label': 'Dữ liệu thiếu'})
plt.title('Heatmap dữ liệu thiếu')
plt.tight_layout()
plt.show()

# Điền giá trị thiếu
# Với biến định tính: thay bằng giá trị yếu vị (mode)
df['DT'].fillna('Kinh', inplace=True)

# Với biến định lượng: thay bằng trung bình hoặc trung vị
df['NGONNGU'].fillna(df['NGONNGU'].mean(), inplace=True)
df['LOGIC'].fillna(df['LOGIC'].mean(), inplace=True)
df['UNGXU'].fillna(df['UNGXU'].median(), inplace=True)

print("\nDữ liệu sau khi xử lý thiếu:")
print(df.head())

# Kiểm tra lại dữ liệu thiếu
print(f"\nDữ liệu thiếu sau khi xử lý:")
missing_after = df.isnull().sum()
print(missing_after[missing_after > 0])


## 2. Feature Engineering

### 2.1. Tạo biến TBTOAN: trung bình toán lớp 12


In [ ]:
# Tạo biến TBTOAN: trung bình toán lớp 12
df['TBTOAN'] = (df['T5'] + df['T6']) / 2

print("Kết quả sau khi tạo biến TBTOAN:")
print(df[['T5', 'T6', 'TBTOAN']].head())

# Tạo biến XEPLOAI: đánh giá môn toán dựa trên df['TBTOAN']
df.loc[df['TBTOAN'] < 5.0, 'XEPLOAI'] = 'FAIL'
df.loc[(df['TBTOAN'] >= 5.0) & (df['TBTOAN'] < 7.0), 'XEPLOAI'] = 'FAIR'
df.loc[(df['TBTOAN'] >= 7.0) & (df['TBTOAN'] < 9.0), 'XEPLOAI'] = 'GOOD'
df.loc[(df['TBTOAN'] >= 9.0), 'XEPLOAI'] = 'EXCEL'

print("\nKết quả sau khi tạo biến XEPLOAI:")
print(df[['TBTOAN', 'XEPLOAI']].head())

# Xem phân bố của XEPLOAI
print("\nPhân bố XEPLOAI:")
print(df['XEPLOAI'].value_counts())


### 2.2. Tạo biến nhóm khối thi NHOMKT


In [ ]:
# Tạo biến nhóm khối thi NHOMKT thỏa mãn:
# A1: G1, C: G3, D1: G3, A: G1, B: G2
dict_map = {'A1': 'G1', 'C': 'G3', 'D1': 'G3', 'A': 'G1', 'B': 'G2'}
df['NHOMKT'] = df['KT'].map(dict_map)

print("Kết quả sau khi tạo biến NHOMKT:")
print(df[['KT', 'NHOMKT']].head())

# Xem phân bố của NHOMKT
print("\nPhân bố NHOMKT:")
print(df['NHOMKT'].value_counts())

# Tạo biến số điểm cộng: CONG
# Nếu khối thi thuộc nhóm G1, G2 và TBTOAN >= 5.0 thì là 1.0
# Ngược lại thì là 0.0
def fplus(x, y):
    if (x == 'G1' or x == 'G2') and (y >= 5.0):
        return 1.0
    else:
        return 0.0

df['CONG'] = list(map(fplus, df['NHOMKT'], df['TBTOAN']))

print("\nKết quả sau khi tạo biến CONG:")
print(df[['TBTOAN', 'NHOMKT', 'CONG']].head())

# Xem phân bố của CONG
print("\nPhân bố CONG:")
print(df['CONG'].value_counts())


## 3. Trực quan hóa dữ liệu

### 3.1. Biểu đồ cột (Bar Plot)


In [ ]:
# Hãy trực quan số lượng học sinh theo giới tính
plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)
sns.countplot(x='GT', data=df)
plt.title('Số lượng học sinh theo giới tính')

# Dựa trên biểu đồ DT cho biết tại sao ta không phân tích theo nhóm DT: vì đa số là dân tộc kinh
plt.subplot(2, 2, 2)
sns.countplot(x='DT', data=df)
plt.title('Số lượng học sinh theo dân tộc')
plt.xticks(rotation=45)

plt.subplot(2, 2, 3)
sns.countplot(x='KV', data=df)
plt.title('Số lượng học sinh theo khu vực')

plt.subplot(2, 2, 4)
sns.countplot(x='KT', data=df)
plt.title('Số lượng học sinh theo khối thi')

plt.tight_layout()
plt.show()


In [ ]:
# Hãy so sánh số lượng học sinh đăng ký khối thi dựa trên nhóm giới tính
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
sns.countplot(x='KT', hue='GT', data=df)
plt.title('Số lượng học sinh đăng ký khối thi theo giới tính')
plt.xticks(rotation=45)

# Hãy cho biết khối A có sinh viên khu vực nào đăng ký cao nhất
plt.subplot(1, 2, 2)
sns.countplot(x='KV', hue='KT', data=df)
plt.title('Số lượng học sinh theo khu vực và khối thi')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

# Hãy so sánh điểm trung bình NGONNGU theo nhóm giới tính
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
sns.barplot(x='GT', y='NGONNGU', data=df, errorbar=None)
plt.title('Điểm trung bình NGONNGU theo giới tính')

# Hãy so sánh điểm LOGIC theo nhóm KT (nhóm khối thi)
plt.subplot(1, 2, 2)
sns.barplot(x='NHOMKT', y='LOGIC', data=df, errorbar=None)
plt.title('Điểm trung bình LOGIC theo nhóm khối thi')

plt.tight_layout()
plt.show()


In [ ]:
# So sánh điểm trung bình của NGONNGU theo nhóm GT dựa trên KT
plt.figure(figsize=(12, 6))
sns.barplot(x='GT', y='NGONNGU', hue='KT', data=df, errorbar=None)
plt.title('Điểm trung bình NGONNGU theo giới tính và khối thi')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# So sánh điểm cao nhất của NGONNGU theo nhóm GT theo KT
plt.figure(figsize=(12, 6))
sns.barplot(x='GT', y='NGONNGU', hue='KT', data=df, errorbar=None, estimator=np.max)
plt.title('Điểm cao nhất NGONNGU theo giới tính và khối thi')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Khi biến định tính dùng làm nhóm tổng hợp có nhiều hơn 2 giá trị thì ta cần dùng hàm tổng hợp thông qua thư viện numpy
plt.figure(figsize=(12, 6))
sns.barplot(x='KV', y='NGONNGU', hue='KT', data=df, errorbar=None, estimator=np.max)
plt.title('Điểm cao nhất NGONNGU theo khu vực và khối thi')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### 3.2. Biểu đồ Pie Chart


In [ ]:
# Biểu đồ PIE - Trực quan dữ liệu theo nhóm tỉ lệ phần trăm
plt.figure(figsize=(12, 6))

# Phân bố theo khối thi
plt.subplot(1, 2, 1)
gb = df.groupby(['KT'])['KT'].agg(['count'])
labels = gb.index
data = list(gb['count'])
colors = sns.color_palette('pastel')
plt.pie(data, labels=labels, colors=colors, autopct='%1.1f%%', shadow=True)
plt.title('Phân bố học sinh theo khối thi')

# Trực quan tỉ lệ % tổng điểm CONG trên từng nhóm khu vực
plt.subplot(1, 2, 2)
gb = df.groupby(['KV'])['CONG'].agg(['sum'])
labels = gb.index
data = list(gb['sum'])
colors = sns.color_palette('pastel')
plt.pie(data, labels=labels, colors=colors, autopct='%1.1f%%', shadow=True)
plt.title('Tỉ lệ điểm cộng theo khu vực')

plt.tight_layout()
plt.show()


### 3.3. Biểu đồ Line Plot


In [ ]:
# Biểu đồ line thường dùng để tổng hợp dữ liệu theo trục "Thời gian" hoặc "có thứ tự"
plt.figure(figsize=(12, 6))

# Trực quan dữ liệu tổng điểm CONG dựa theo năm bằng biểu đồ line
plt.subplot(1, 2, 1)
sns.lineplot(x='NGAYTHI', y='CONG', data=df)
plt.title('Điểm cộng trung bình theo năm thi')

# Tổng hợp tổng điểm cộng theo các năm thi trên từng nhóm giới tính bằng biểu đồ line
plt.subplot(1, 2, 2)
sns.lineplot(x='NGAYTHI', y='CONG', hue='GT', data=df, estimator=sum)
plt.title('Tổng điểm cộng theo năm thi và giới tính')

plt.tight_layout()
plt.show()


## 4. Mô tả dữ liệu định lượng


In [ ]:
# Mô tả dữ liệu cột NGONNGU
print("Mô tả dữ liệu cột NGONNGU:")
print(df['NGONNGU'].describe())

# Mô tả dữ liệu của NGONNGU, LOGIC, UNGXU
print("\nMô tả dữ liệu của NGONNGU, LOGIC, UNGXU:")
print(df[['NGONNGU', 'LOGIC', 'UNGXU']].describe())

# Mô tả dữ liệu theo nhóm giới tính
print("\nMô tả dữ liệu theo nhóm giới tính:")
print(df.groupby('GT')[['NGONNGU', 'LOGIC', 'UNGXU']].describe())

# CV = std/mean (Coefficient of variant)
# So sánh mức độ ổn định của điểm số
cvNN = df['NGONNGU'].std() / df['NGONNGU'].mean()
cvLogic = df['LOGIC'].std() / df['LOGIC'].mean()
cvUngXu = df['UNGXU'].std() / df['UNGXU'].mean()

print(f"\nHệ số biến thiên (CV):")
print(f"CV NGONNGU: {cvNN:.4f}")
print(f"CV LOGIC: {cvLogic:.4f}")
print(f"CV UNGXU: {cvUngXu:.4f}")

# Cách tính CV cho tất cả cột
cv_all = df[['NGONNGU', 'LOGIC', 'UNGXU']].std() / df[['NGONNGU', 'LOGIC', 'UNGXU']].mean()
print(f"\nCV cho tất cả các cột:")
print(cv_all)


## 5. Phân tích phân phối dữ liệu


In [ ]:
# Histogram cho biết xác suất xảy ra của biến cố trong khoảng giá trị dữ liệu nào nhiều nhất
plt.figure(figsize=(15, 10))

# Histogram cơ bản
plt.subplot(2, 3, 1)
df['NGONNGU'].hist(bins=20)
plt.title('Histogram NGONNGU (bins=20)')
plt.xlabel('Điểm')
plt.ylabel('Tần suất')

# Histogram với bins khác nhau
plt.subplot(2, 3, 2)
df['NGONNGU'].hist(bins=14)
plt.title('Histogram NGONNGU (bins=14)')
plt.xlabel('Điểm')
plt.ylabel('Tần suất')

# Nâng cao hơn histogram thì ra khám phá dạng phân phối xác xuất
plt.subplot(2, 3, 3)
sns.displot(df, x='NGONNGU', kind='kde')
plt.title('Phân phối NGONNGU với KDE')

# Phân phối của nhiều biến
plt.subplot(2, 3, 4)
sns.displot(data=df[['NGONNGU', 'LOGIC', 'UNGXU']], kind='kde')
plt.title('Phân phối NGONNGU, LOGIC, UNGXU')

# Phân phối theo giới tính
plt.subplot(2, 3, 5)
sns.displot(df, x='NGONNGU', hue='GT', kind='kde')
plt.title('Phân phối NGONNGU theo giới tính')

plt.tight_layout()
plt.show()


In [ ]:
# Boxplot - Biểu đồ quan trọng trong việc phân tích dữ liệu định lượng
plt.figure(figsize=(15, 10))

# Boxplot cơ bản
plt.subplot(2, 3, 1)
sns.boxplot(data=df['LOGIC'], orient="h")
plt.title('Boxplot LOGIC')

# Tính các giá trị quan trọng
IQR = df['LOGIC'].quantile(0.75) - df['LOGIC'].quantile(0.25)
lower_bound = df['LOGIC'].quantile(0.25) - 1.5 * IQR
upper_bound = df['LOGIC'].quantile(0.75) + 1.5 * IQR

print(f"Lower bound = {lower_bound:.3f}")
print(f"Upper bound = {upper_bound:.3f}")
print(f"IQR = {IQR:.3f}")

# Boxplot nhiều biến
plt.subplot(2, 3, 2)
sns.boxplot(data=df[['NGONNGU', 'LOGIC', 'UNGXU']], orient='h')
plt.title('Boxplot NGONNGU, LOGIC, UNGXU')

# Boxplot theo khối thi
plt.subplot(2, 3, 3)
sns.boxplot(x='NGONNGU', y='KT', data=df, orient='h')
plt.title('Boxplot NGONNGU theo khối thi')

# Boxplot theo khu vực
plt.subplot(2, 3, 4)
sns.boxplot(x='NGONNGU', y='KV', data=df, orient='h')
plt.title('Boxplot NGONNGU theo khu vực')

# Boxplot theo giới tính và khối thi
plt.subplot(2, 3, 5)
sns.boxplot(x='KT', y='NGONNGU', hue='GT', data=df)
plt.title('Boxplot NGONNGU theo khối thi và giới tính')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
# Skewness = độ xiên, độ lớn (trị tuyệt đối) cho biết mức độ dữ liệu lệch nhiều hay ít so với đường cong phân phối chuẩn
print("Độ xiên của phân phối:")
print(df['NGONNGU'].skew())

print("\nĐộ xiên của các biến:")
print(df[['NGONNGU', 'LOGIC', 'UNGXU']].skew())

# Kurtosis: Độ nhọn, trị tuyệt đối cho biết mức độ nhọn của phân phối
print("\nĐộ nhọn của phân phối:")
print(df[['NGONNGU', 'LOGIC', 'UNGXU']].kurtosis())

# Kiểm định phân phối chuẩn
plt.figure(figsize=(10, 6))
stats.probplot(df['NGONNGU'], plot=plt)
plt.title('Q-Q Plot kiểm tra phân phối chuẩn NGONNGU')
plt.show()

# Phân tích sự tác động (ảnh hưởng) qua lại giữa 2 biến định lượng
print("\nMa trận hiệp phương sai:")
print(df[['T5', 'T6']].cov())

print("\nMa trận hiệp phương sai mở rộng:")
print(df[['T5', 'T6', 'LOGIC']].cov())


## 6. Phân tích tương quan


In [ ]:
# Pearson Correlation: tương quan tuyến tính
print("Ma trận tương quan T5 và T6:")
print(df[['T5', 'T6']].corr())

print("\nMa trận tương quan mở rộng:")
print(df[['T5', 'T6', 'LOGIC', 'UNGXU', 'NGONNGU']].corr())

# Trực quan hóa tương quan tuyến tính giữa 2 biến định lượng
plt.figure(figsize=(12, 8))

# Scatter plot với đường hồi quy
plt.subplot(2, 2, 1)
sns.lmplot(data=df, x='T5', y='T6', fit_reg=True)
plt.title('Tương quan T5 và T6')

# Tương quan T6 và UNGXU
plt.subplot(2, 2, 2)
sns.lmplot(data=df, x='T6', y='UNGXU', fit_reg=True)
plt.title('Tương quan T6 và UNGXU')

# Heatmap tương quan
plt.subplot(2, 2, 3)
correlation_matrix = df[['T5', 'T6', 'UNGXU', 'NGONNGU', 'LOGIC']].corr()
sns.heatmap(correlation_matrix, vmax=1.0, square=False, annot=True, cmap='coolwarm', center=0)
plt.title('Ma trận tương quan')

# Pairplot
plt.subplot(2, 2, 4)
sns.pairplot(df[['T5', 'T6', 'NGONNGU', 'LOGIC', 'UNGXU']], diag_kind='kde', kind='reg')
plt.suptitle('Pairplot các biến định lượng', y=1.02)

plt.tight_layout()
plt.show()


In [ ]:
# Trực quan tương quan tuyến tính theo nhóm (định tính) giữa 2 biến định lượng
plt.figure(figsize=(12, 6))
sns.lmplot(data=df, x='T5', y='T6', hue='GT', fit_reg=True)
plt.title('Tương quan T5 và T6 theo giới tính')
plt.show()

# Lưu dữ liệu đã xử lý
df.to_csv('processed_tuyen_sinh_data.csv', sep=',', encoding='utf-8', index=False)
print("Đã lưu dữ liệu đã xử lý vào file 'processed_tuyen_sinh_data.csv'")

# Thống kê cuối cùng
print(f"\nThông tin cuối cùng về DataFrame:")
print(f"Số dòng: {len(df)}")
print(f"Số cột: {len(df.columns)}")
print(f"Các cột: {list(df.columns)}")

print("\nTóm tắt kết quả phân tích:")
print("1. Đã xử lý dữ liệu thiếu và tạo các biến mới")
print("2. Đã thực hiện trực quan hóa dữ liệu với nhiều loại biểu đồ")
print("3. Đã phân tích phân phối dữ liệu và tương quan giữa các biến")
print("4. Đã xác định được các mối quan hệ trong dữ liệu tuyển sinh")
